# 20D skew-symmetric product-form benchmark

This notebook is the compact, self-contained reference implementation of the
20-dimensional Holomorphic Cauchy Network (HCN) experiment.  It reproduces the
fixed-capacity architecture and the fixed 25,000-update, three-seed protocol
used for the paper's confirmatory result.

The training path uses only the RBM basic adjoint relationship (BAR), boundary
consistency, and transform-shape constraints.  Exact tails and NN4MGF values
are introduced only after all fixed endpoints have been written.  Intermediate
checkpoints are neither evaluated nor ranked.  Predictions are reported raw,
without clipping.

The safe default is `RUN_MODE = "smoke"`; a clean **Run All** therefore performs
only structural checks and two tiny optimizer steps.  Use `RUN_MODE =
"train_one"` to run one prespecified seed, `RUN_MODE = "train_all"` for all
three sequentially, or `RUN_MODE = "evaluate"` with a completed run directory.


## 1. Imports and reproducibility


In [ ]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import hashlib
import json
import math
import platform
import random
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
r_dtype = torch.float64
c_dtype = torch.complex128


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        warnings.warn(f"Could not enable deterministic algorithms: {exc}")


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if not rows and fieldnames is None:
        return
    fields = list(fieldnames or rows[0].keys())
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


set_all_seeds(12345)
print({"device": str(device), "torch": torch.__version__, "complex_dtype": str(c_dtype)})


## 2. Frozen paper configuration


In [ ]:
# Public defaults are safe: Run All performs only a two-step smoke test.
RUN_MODE = "smoke"                 # "smoke", "train_one", "train_all", "evaluate"
TRAIN_SEED = 12345                  # used only when RUN_MODE == "train_one"
EVALUATE_RUN_DIR = None            # required when RUN_MODE == "evaluate"

ALLOWED_RUN_MODES = {"smoke", "train_one", "train_all", "evaluate"}
if RUN_MODE not in ALLOWED_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(ALLOWED_RUN_MODES)}")
if RUN_MODE == "train_one" and TRAIN_SEED not in (12345, 22345, 32345):
    raise ValueError("TRAIN_SEED must be one of the three prespecified paper seeds.")
if RUN_MODE == "evaluate" and EVALUATE_RUN_DIR is None:
    raise ValueError("Set EVALUATE_RUN_DIR explicitly before using RUN_MODE='evaluate'.")

EXPERIMENT_SLUG = "d20_skew_symmetric_product_form"
OUTPUT_ROOT = Path("outputs")
DIMENSION = 20
PAPER_SEEDS = (12345, 22345, 32345)
TRAIN_STEPS = 25_000
LR_SWITCH_AFTER_STEP = 20_000
CHECKPOINT_STEP = 25_000
LOG_EVERY = 1_000

MODEL_CONFIG = {
    "d": DIMENSION,
    "coord_embed_dim": 16,
    "boundary_embed_dim": 16,
    "interior_hidden_width": 128,
    "boundary_hidden_width": 128,
    "pair_chunk_size": 32_768,
    "boundary_anchor_mode": "hard_rbm_flow_balance",
}

TRAIN_CONFIG = {
    "optimizer": "Adam",
    "steps": TRAIN_STEPS,
    "batch_bar": 2_048,
    "batch_boundary": 256,
    "batch_monotonicity": 256,
    "lr_phase1": 5.0e-5,
    "lr_phase2": 1.0e-5,
    "lr_switch_after_step": LR_SWITCH_AFTER_STEP,
    "preserve_adam_state_at_lr_switch": True,
    "gradient_clip": 1.0,
    "lambda_bar": 1.0,
    "lambda_boundary": 1.0,
    "lambda_origin": 0.0,
    "lambda_monotonicity": 1.0,
    "lambda_imaginary": 1.0,
    "monotonicity_real_domain": [-0.5, 5.0],
    "adaptive_bar_weight": {
        "enabled": True,
        "alpha": 0.3,
        "epsilon": 1.0e-12,
        "minimum": 0.5,
        "maximum": 2.0,
    },
}

SAMPLER_CONFIG = {
    "real_domain": [-0.5, 5.0],
    "imaginary_abs_max": 5.0,
    "bar_mixture": {
        "general_complex": 0.50,
        "exact_complex_diagonal": 0.20,
        "near_complex_diagonal": 0.20,
        "positive_real_diagonal": 0.10,
    },
    "near_diagonal_noise_scale": 0.05,
}

EVALUATION_CONFIG = {
    "t_values": list(range(1, 11)),
    "talbot_degrees": [5, 8],
    "primary_degree": 8,
    "degree5_role": "numerical-inversion diagnostic and published-comparison bridge",
    "raw_predictions": True,
    "prediction_clipping": False,
    "mpmath_decimal_digits": 100,
}

RUN_CONFIG = {
    "experiment_slug": EXPERIMENT_SLUG,
    "dimension": DIMENSION,
    "paper_seeds": list(PAPER_SEEDS),
    "model": MODEL_CONFIG,
    "training": TRAIN_CONFIG,
    "sampler": SAMPLER_CONFIG,
    "evaluation": EVALUATION_CONFIG,
    "checkpoint_selection": "prespecified fixed step 25000 only",
    "intermediate_checkpoint_evaluation": False,
    "ground_truth_used_by_training": False,
    "ground_truth_used_for_checkpoint_selection": False,
}

PER_SEED_SAMPLE_BUDGET = TRAIN_STEPS * (
    TRAIN_CONFIG["batch_bar"]
    + TRAIN_CONFIG["batch_boundary"]
    + TRAIN_CONFIG["batch_monotonicity"]
)
SAMPLE_BUDGET = {
    "per_seed": {
        "bar_points": TRAIN_STEPS * TRAIN_CONFIG["batch_bar"],
        "boundary_points": TRAIN_STEPS * TRAIN_CONFIG["batch_boundary"],
        "monotonicity_points": TRAIN_STEPS * TRAIN_CONFIG["batch_monotonicity"],
        "total_sampled_constraints": PER_SEED_SAMPLE_BUDGET,
    },
    "number_of_seeds": len(PAPER_SEEDS),
    "all_seeds_total_sampled_constraints": len(PAPER_SEEDS) * PER_SEED_SAMPLE_BUDGET,
}
assert PER_SEED_SAMPLE_BUDGET == 64_000_000
print({
    "run_mode": RUN_MODE,
    "train_seed": TRAIN_SEED if RUN_MODE == "train_one" else None,
    "paper_seeds": PAPER_SEEDS,
    "per_seed_budget": PER_SEED_SAMPLE_BUDGET,
})


## 3. RBM benchmark and BAR coefficients


In [ ]:
@dataclass
class RBMParams:
    mu: torch.Tensor
    Sigma: torch.Tensor
    R: torch.Tensor
    alpha: torch.Tensor
    delta: torch.Tensor | None = None
    a: torch.Tensor | None = None


def make_skew_symmetric_product_form_params(d, device=device, r_dtype=r_dtype, c_dtype=c_dtype):
    R = torch.zeros(d, d, dtype=r_dtype, device=device)
    Sigma = torch.zeros(d, d, dtype=r_dtype, device=device)
    idx = torch.arange(d, device=device)
    R[idx, idx] = 1.0
    Sigma[idx, idx] = 2.0
    for i in range(1, d):
        R[i, i - 1] = -1.0
        Sigma[i, i - 1] = -1.0
        Sigma[i - 1, i] = -1.0

    mu = -torch.ones(d, dtype=r_dtype, device=device)
    alpha = torch.arange(1, d + 1, dtype=r_dtype, device=device)
    alpha_formula = -2.0 * torch.diag(R) * torch.linalg.solve(R, mu) / torch.diag(Sigma)
    if not torch.allclose(alpha_formula, alpha, rtol=1e-12, atol=1e-12):
        raise AssertionError("Formula alpha check failed.")
    delta = torch.diag(Sigma) / (2.0 * torch.diag(R)) * alpha
    return RBMParams(mu=mu, Sigma=Sigma, R=R, alpha=alpha, delta=delta, a=alpha)


def compute_boundary_mass(params, tol=1e-12):
    mass = -torch.linalg.solve(
        params.R.to(dtype=r_dtype, device=device),
        params.mu.to(dtype=r_dtype, device=device),
    )
    if mass.ndim != 1:
        raise ValueError(f"Boundary mass must be one-dimensional, got shape {tuple(mass.shape)}")
    if not torch.isfinite(mass).all():
        raise ValueError("Boundary mass contains non-finite entries.")
    if torch.any(mass <= tol):
        raise ValueError(
            "Boundary rates must be strictly positive. "
            "Check RBM stability and the R/mu sign convention."
        )
    return mass


def compute_gammas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    mu_c = params.mu.to(dtype=c_dtype, device=device)
    Sigma_c = params.Sigma.to(dtype=c_dtype, device=device)
    R_c = params.R.to(dtype=c_dtype, device=device)
    gamma0 = -0.5 * torch.sum((theta @ Sigma_c) * theta, dim=1, keepdim=True) + theta @ mu_c.view(-1, 1)
    gamma_all = -(theta @ R_c)  # column-based convention
    return gamma0, gamma_all


def complex_log_safe(z, eps=1e-14):
    mag = torch.clamp(torch.abs(z), min=eps)
    return torch.complex(torch.log(mag), torch.angle(z))


PARAMS = make_skew_symmetric_product_form_params(DIMENSION)
BOUNDARY_MASS = compute_boundary_mass(PARAMS)
EXPECTED_BOUNDARY_MASS = torch.arange(1, DIMENSION + 1, dtype=r_dtype, device=device)
if not torch.allclose(BOUNDARY_MASS, EXPECTED_BOUNDARY_MASS, atol=1e-12, rtol=0.0):
    raise AssertionError("The audited 20D boundary masses must equal rates 1,...,20.")

RBM_AUDIT = {
    "dimension": int(PARAMS.mu.numel()),
    "mu": PARAMS.mu.detach().cpu().tolist(),
    "sigma": PARAMS.Sigma.detach().cpu().tolist(),
    "reflection_matrix": PARAMS.R.detach().cpu().tolist(),
    "boundary_mass": BOUNDARY_MASS.detach().cpu().tolist(),
}
print({"dimension": int(PARAMS.mu.numel()), "boundary_mass": BOUNDARY_MASS.detach().cpu().tolist()})


## 4. Coordinate-shared Holomorphic Cauchy Network

The interior transform and the 20 boundary transforms share two scalar
holomorphic Cauchy networks.  Coordinate and boundary embeddings identify the
requested term without introducing a separate network per coordinate.  Origin
normalization and boundary masses are imposed by construction.


In [ ]:
class CauchyComplex(nn.Module):
    def __init__(self, num_features, init_lambda=0.1, init_d=0.5, d_min=1e-3):
        super().__init__()
        self.num_features = num_features
        self.d_min = d_min
        self.lambda1 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        self.lambda2 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        init_value = max(init_d - d_min, 1e-6)
        init_rho = torch.log(torch.expm1(torch.tensor(init_value, dtype=r_dtype)))
        self.rho = nn.Parameter(torch.ones(num_features, dtype=r_dtype) * init_rho)
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def positive_d(self):
        return F.softplus(self.rho) + self.d_min

    def reset_denominator_tracking(self):
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def _update_denominator_tracking(self, denom):
        current = torch.min(torch.abs(denom)).detach()
        self.last_min_abs_denominator = current
        if self.running_min_abs_denominator is None:
            self.running_min_abs_denominator = current
        else:
            self.running_min_abs_denominator = torch.minimum(self.running_min_abs_denominator, current).detach()

    def _denominator(self, z):
        d_complex = self.positive_d().to(device=z.device).to(dtype=z.dtype)
        return z ** 2 + d_complex ** 2, d_complex

    def forward(self, z):
        denom, _ = self._denominator(z)
        self._update_denominator_tracking(denom)
        out = (self.lambda1 * z + self.lambda2) / denom
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError("Non-finite Cauchy activation output.")
        return out

    def derivative(self, z):
        denom, d_complex = self._denominator(z)
        self._update_denominator_tracking(denom)
        out = (self.lambda1 * (d_complex ** 2 - z ** 2) - 2.0 * self.lambda2 * z) / (denom ** 2)
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError("Non-finite Cauchy derivative output.")
        return out


def reset_all_cauchy_denominator_tracking(model):
    for module in model.modules():
        if isinstance(module, CauchyComplex):
            module.reset_denominator_tracking()


class CoordinateCauchyScalarNet(nn.Module):
    """
    Shared scalar complex network:
        Linear(input_dim,H) -> CauchyComplex(H) -> Linear(H,1).
    First input column is the scalar coordinate theta_i.
    """

    def __init__(self, input_dim, hidden_width):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_width = hidden_width
        self.linear_0 = nn.Linear(input_dim, hidden_width, dtype=c_dtype)
        self.cauchy_0 = CauchyComplex(hidden_width)
        self.linear_1 = nn.Linear(hidden_width, 1, dtype=c_dtype)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError(f"x must have shape [N,{self.input_dim}], got {tuple(x.shape)}")
        x = x.to(dtype=c_dtype, device=device)
        return self.linear_1(self.cauchy_0(self.linear_0(x)))

    def forward_with_first_input_derivative(self, x):
        x = x.to(dtype=c_dtype, device=device)
        z = self.linear_0(x)                    # [N,H]
        phi = self.cauchy_0(z)                  # [N,H]
        out = self.linear_1(phi)                # [N,1]
        phi_prime = self.cauchy_0.derivative(z) # [N,H]
        # derivative wrt x[:,0]: sum_h W_out[0,h] Phi'_h W_in[h,0]
        deriv = torch.einsum("h,nh,h->n", self.linear_1.weight[0], phi_prime, self.linear_0.weight[:, 0]).unsqueeze(1)
        return out, deriv


def run_scalar_net_chunked(net, x, pair_chunk_size=32768, with_derivative=False):
    outs = []
    derivs = []
    for start in range(0, x.shape[0], pair_chunk_size):
        chunk = x[start:start + pair_chunk_size]
        if with_derivative:
            out, deriv = net.forward_with_first_input_derivative(chunk)
            outs.append(out)
            derivs.append(deriv)
        else:
            outs.append(net(chunk))
    if with_derivative:
        return torch.cat(outs, dim=0), torch.cat(derivs, dim=0)
    return torch.cat(outs, dim=0)


class StructuredCoordinateCauchyMGF(nn.Module):
    """
    Coordinate-wise additive Cauchy model.
    Python k is zero-based and represents mathematical boundary k+1.
    Boundary terms are zero-centered so f_k(0)=log(boundary_mass[k]).
    """

    def __init__(
        self,
        d=20,
        coord_embed_dim=16,
        boundary_embed_dim=16,
        interior_hidden_width=128,
        boundary_hidden_width=128,
        pair_chunk_size=32768,
        boundary_mass=None,
    ):
        super().__init__()
        self.d = int(d)
        self.coord_embed_dim = int(coord_embed_dim)
        self.boundary_embed_dim = int(boundary_embed_dim)
        self.pair_chunk_size = int(pair_chunk_size)
        if boundary_mass is None:
            raise ValueError("StructuredCoordinateCauchyMGF requires boundary_mass.")
        boundary_mass_real = boundary_mass.detach().clone().to(dtype=r_dtype, device=device)
        if boundary_mass_real.shape != (self.d,):
            raise ValueError(f"boundary_mass must have shape ({self.d},), got {tuple(boundary_mass_real.shape)}")
        if not torch.isfinite(boundary_mass_real).all():
            raise ValueError("boundary_mass contains non-finite entries.")
        if torch.any(boundary_mass_real <= 0):
            raise ValueError("boundary_mass entries must be strictly positive.")
        self.register_buffer("boundary_mass", boundary_mass_real, persistent=True)
        self.register_buffer("boundary_log_mass", torch.log(boundary_mass_real).to(c_dtype), persistent=True)
        scale = 0.01
        self.coord_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype),
                torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype),
            )
        )
        self.boundary_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype),
                torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype),
            )
        )
        self.interior_net = CoordinateCauchyScalarNet(1 + self.coord_embed_dim, interior_hidden_width)
        self.boundary_net = CoordinateCauchyScalarNet(1 + self.coord_embed_dim + self.boundary_embed_dim, boundary_hidden_width)

    def _interior_inputs(self, theta):
        B, d = theta.shape
        coord = self.coord_embedding.view(1, d, self.coord_embed_dim).expand(B, d, self.coord_embed_dim)
        return torch.cat([theta.unsqueeze(-1), coord], dim=2).reshape(B * d, 1 + self.coord_embed_dim)

    def _interior_f0_and_jac(self, theta, need_jacobian):
        B, d = theta.shape
        x = self._interior_inputs(theta)
        if need_jacobian:
            q, dq = run_scalar_net_chunked(self.interior_net, x, self.pair_chunk_size, with_derivative=True)
            dq = dq.view(B, d)
        else:
            q = run_scalar_net_chunked(self.interior_net, x, self.pair_chunk_size, with_derivative=False)
            dq = None
        q = q.view(B, d)
        zero_theta = torch.zeros(d, 1, dtype=c_dtype, device=theta.device)
        x0 = torch.cat([zero_theta, self.coord_embedding], dim=1)
        q0 = run_scalar_net_chunked(self.interior_net, x0, self.pair_chunk_size, with_derivative=False).view(1, d)
        f0 = torch.sum(q - q0, dim=1, keepdim=True)
        return f0, dq

    def _boundary_inputs_for_k(self, theta, k):
        B, d = theta.shape
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        theta_sel = theta[:, mask]
        coord = self.coord_embedding[mask].view(1, d - 1, self.coord_embed_dim).expand(B, d - 1, self.coord_embed_dim)
        bd = self.boundary_embedding[k].view(1, 1, self.boundary_embed_dim).expand(B, d - 1, self.boundary_embed_dim)
        x = torch.cat([theta_sel.unsqueeze(-1), coord, bd], dim=2).reshape(B * (d - 1), 1 + self.coord_embed_dim + self.boundary_embed_dim)
        return x, mask

    def _boundary_zero_inputs_for_k(self, theta, k):
        d = self.d
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        zeros = torch.zeros(d - 1, 1, dtype=c_dtype, device=theta.device)
        coord = self.coord_embedding[mask]
        bd = self.boundary_embedding[k].view(1, self.boundary_embed_dim).expand(d - 1, self.boundary_embed_dim)
        return torch.cat([zeros, coord, bd], dim=1)

    def _boundary_fk_and_deriv_for_k(self, theta, k, need_derivative):
        B, d = theta.shape
        x, mask = self._boundary_inputs_for_k(theta, k)
        x0 = self._boundary_zero_inputs_for_k(theta, k)
        if need_derivative:
            q, dq = run_scalar_net_chunked(self.boundary_net, x, self.pair_chunk_size, with_derivative=True)
            dq = dq.view(B, d - 1)
        else:
            q = run_scalar_net_chunked(self.boundary_net, x, self.pair_chunk_size, with_derivative=False)
            dq = None
        q0 = run_scalar_net_chunked(self.boundary_net, x0, self.pair_chunk_size, with_derivative=False).view(1, d - 1)
        q = q.view(B, d - 1)
        fk = self.boundary_log_mass[k].view(1, 1) + torch.sum(q - q0, dim=1, keepdim=True)
        return fk, dq, mask

    def forward_selected_boundary(self, theta, k):
        theta = theta.to(dtype=c_dtype, device=device)
        if k < 0 or k >= self.d:
            raise IndexError(f"Boundary index k={k} outside [0,{self.d-1}]")
        f0, _ = self._interior_f0_and_jac(theta, need_jacobian=False)
        fk, _, _ = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=False)
        return f0, fk

    def forward(self, theta):
        f_mat, _ = self.forward_with_input_jacobian(theta, need_jacobian=False)
        return f_mat

    def forward_with_input_jacobian(self, theta, need_jacobian=True):
        theta = theta.to(dtype=c_dtype, device=device)
        if theta.ndim != 2 or theta.shape[1] != self.d:
            raise ValueError(f"theta must have shape [B,{self.d}], got {tuple(theta.shape)}")
        B, d = theta.shape
        f0, dq_int = self._interior_f0_and_jac(theta, need_jacobian=need_jacobian)
        boundary_cols = []
        if need_jacobian:
            J = torch.zeros(B, d + 1, d, dtype=c_dtype, device=theta.device)
            J[:, 0, :] = dq_int
        else:
            J = None
        for k in range(d):
            fk, dq, mask = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=need_jacobian)
            boundary_cols.append(fk)
            if need_jacobian:
                J[:, k + 1, mask] = dq
        return torch.cat([f0] + boundary_cols, dim=1), J


def count_complex_parameters(model):
    complex_count = sum(p.numel() for p in model.parameters() if p.is_complex())
    real_count = sum(p.numel() for p in model.parameters() if not p.is_complex())
    real_degrees = 2 * complex_count + real_count
    return {"complex_params": complex_count, "real_params": real_count, "real_degrees": real_degrees}


def min_cauchy_denominator(net):
    vals = []
    for module in net.modules():
        if isinstance(module, CauchyComplex) and module.running_min_abs_denominator is not None:
            vals.append(float(module.running_min_abs_denominator.detach().cpu()))
    return min(vals) if vals else float("nan")


def build_model(seed):
    set_all_seeds(int(seed))
    return StructuredCoordinateCauchyMGF(
        d=MODEL_CONFIG["d"],
        coord_embed_dim=MODEL_CONFIG["coord_embed_dim"],
        boundary_embed_dim=MODEL_CONFIG["boundary_embed_dim"],
        interior_hidden_width=MODEL_CONFIG["interior_hidden_width"],
        boundary_hidden_width=MODEL_CONFIG["boundary_hidden_width"],
        pair_chunk_size=MODEL_CONFIG["pair_chunk_size"],
        boundary_mass=BOUNDARY_MASS,
    ).to(device)


def parameter_audit(model):
    basic = count_complex_parameters(model)
    modules = []
    for name, parameter in model.named_parameters():
        tensor_elements = int(parameter.numel())
        real_dof = tensor_elements * (2 if parameter.is_complex() else 1)
        modules.append({
            "name": name,
            "dtype": str(parameter.dtype),
            "tensor_elements": tensor_elements,
            "real_scalar_dof": real_dof,
            "trainable": bool(parameter.requires_grad),
        })
    return {
        "trainable_tensor_elements": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "complex_parameter_elements": int(basic["complex_params"]),
        "real_parameter_elements": int(basic["real_params"]),
        "trainable_real_scalar_dof": int(basic["real_degrees"]),
        "modules": modules,
    }


REFERENCE_MODEL = build_model(PAPER_SEEDS[0])
PARAMETER_AUDIT = parameter_audit(REFERENCE_MODEL)
assert PARAMETER_AUDIT["trainable_tensor_elements"] == 8_322
assert PARAMETER_AUDIT["trainable_real_scalar_dof"] == 16_388
print({k: PARAMETER_AUDIT[k] for k in (
    "trainable_tensor_elements", "trainable_real_scalar_dof"
)})


## 5. Complex-domain mixed sampler


In [ ]:
def sample_complex_box(N, d=20, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    real = re_min + (re_max - re_min) * torch.rand(N, d, dtype=r_dtype, device=device)
    imag = -im_max + 2.0 * im_max * torch.rand(N, d, dtype=r_dtype, device=device)
    theta = torch.complex(real, imag)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_nn4mgf_vector_mgf(N, d=20, lb=-5.0, ub=0.5, imag_lb=-5.0, imag_ub=5.0, device=device):
    real_lb_curr = ub - (ub - lb) * torch.rand(N, 1, dtype=r_dtype, device=device)
    imag_bd = max(abs(imag_ub), abs(imag_lb)) * torch.rand(N, 1, dtype=r_dtype, device=device)
    real = (ub - real_lb_curr) * torch.rand(N, d, dtype=r_dtype, device=device) + real_lb_curr
    imag = 2.0 * imag_bd * torch.rand(N, d, dtype=r_dtype, device=device) - imag_bd
    return torch.complex(real, imag)


def sample_nn4mgf_vector_laplace(N, d=20, lap_re_min=-0.5, lap_re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    theta = -sample_nn4mgf_vector_mgf(N=N, d=d, lb=-lap_re_max, ub=-lap_re_min, imag_lb=-im_max, imag_ub=im_max, device=device)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_training_theta(N, d=20, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    return sample_nn4mgf_vector_laplace(N=N, d=d, lap_re_min=re_min, lap_re_max=re_max, im_max=im_max, requires_grad=requires_grad, device=device)


def sample_nested_real_radius(N, lb=0.0, ub=5.0, device=device):
    # Near-zero-biased sampler: E[s] = lb + (ub-lb)/3 for U(0,1)^2.
    u = torch.rand(N, 1, dtype=r_dtype, device=device)
    return lb + (ub - lb) * (u ** 2)


def sample_mixed_bar_theta(N=4000, d=20, re_min=-0.5, re_max=5.0, im_max=5.0, device=device):
    """
    Mixed BAR sampler proportions:
      50% general, 20% exact complex diagonal, 20% near-diagonal complex, 10% positive-real diagonal.
    Counts are scaled proportionally for N != 4000.
    """
    n_global = int(round(N * 0.50))
    n_exact_diag = int(round(N * 0.20))
    n_near_diag = int(round(N * 0.20))
    n_pos_diag = N - n_global - n_exact_diag - n_near_diag

    global_theta = sample_training_theta(n_global, d=d, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    s_complex = sample_training_theta(n_exact_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    exact_diag = s_complex.expand(n_exact_diag, d)

    s_near = sample_training_theta(n_near_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    noise = 0.05 * sample_complex_box(n_near_diag, d=d, re_min=-1.0, re_max=1.0, im_max=1.0, device=device)
    near_diag_raw = s_near.expand(n_near_diag, d) + noise
    near_diag = torch.complex(
        torch.clamp(near_diag_raw.real, min=re_min, max=re_max),
        torch.clamp(near_diag_raw.imag, min=-im_max, max=im_max),
    )

    positive_lb = max(0.0, re_min)
    positive_ub = re_max
    if positive_ub <= positive_lb:
        raise ValueError(f"Positive-real diagonal domain is empty: [{positive_lb}, {positive_ub}]")
    s_real = sample_nested_real_radius(n_pos_diag, lb=positive_lb, ub=positive_ub, device=device)
    pos_diag = torch.complex(s_real.expand(n_pos_diag, d), torch.zeros(n_pos_diag, d, dtype=r_dtype, device=device))

    theta = torch.cat([global_theta, exact_diag, near_diag, pos_diag], dim=0).to(dtype=c_dtype, device=device)
    return theta[torch.randperm(theta.shape[0], device=device)]


def mixture_counts(batch_size):
    n_general = int(round(batch_size * 0.50))
    n_exact = int(round(batch_size * 0.20))
    n_near = int(round(batch_size * 0.20))
    n_positive = batch_size - n_general - n_exact - n_near
    return {
        "general_complex": n_general,
        "exact_complex_diagonal": n_exact,
        "near_complex_diagonal": n_near,
        "positive_real_diagonal": n_positive,
    }


ACTUAL_BAR_MIXTURE_COUNTS = mixture_counts(TRAIN_CONFIG["batch_bar"])
assert ACTUAL_BAR_MIXTURE_COUNTS == {
    "general_complex": 1024,
    "exact_complex_diagonal": 410,
    "near_complex_diagonal": 410,
    "positive_real_diagonal": 204,
}
print(ACTUAL_BAR_MIXTURE_COUNTS)


## 6. Unsupervised BAR objective


In [ ]:
DEFAULT_ADAPTIVE_BAR_CONFIG = {
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1.0e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
}

def construct_boundary_directions(params):
    R_c = params.R.to(dtype=c_dtype, device=device)
    A = R_c.T
    d = R_c.shape[0]
    eye = torch.eye(d, dtype=c_dtype, device=device)
    rows = []
    for k in range(d):
        M = A.clone()
        M[k, :] = eye[k, :]
        rows.append(torch.linalg.solve(M, eye[k, :]).view(1, d))
    return torch.cat(rows, dim=0)


def construct_boundary_thetas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    V = construct_boundary_directions(params)
    return [theta[:, k:k+1] * V[k:k+1, :] for k in range(d)], V


def with_adaptive_bar_defaults(config):
    merged = dict(DEFAULT_ADAPTIVE_BAR_CONFIG)
    merged.update(config)
    return merged


def adaptive_bar_kwargs(config):
    cfg = with_adaptive_bar_defaults(config)
    return {
        "adaptive_bar_weight": bool(cfg["adaptive_bar_weight"]),
        "bar_weight_alpha": float(cfg["bar_weight_alpha"]),
        "bar_weight_eps": float(cfg["bar_weight_eps"]),
        "bar_weight_min": float(cfg["bar_weight_min"]),
        "bar_weight_max": float(cfg["bar_weight_max"]),
    }


def split_f_tuple(f_out):
    if isinstance(f_out, torch.Tensor):
        return f_out
    if isinstance(f_out, (tuple, list)):
        return torch.cat(list(f_out), dim=1)
    raise TypeError(f"Unsupported model output type {type(f_out)!r}")


def compute_bar_pointwise(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8):
    f_mat = split_f_tuple(f_tuple)
    gamma0, gamma_all = compute_gammas(theta, params)
    gamma_mat = torch.cat([gamma0, gamma_all], dim=1)
    nu = torch.max(torch.log(torch.abs(gamma_mat) + eps_gamma) + f_mat.real, dim=1, keepdim=True)[0]
    kappa = gamma_mat * torch.exp(f_mat - nu.to(dtype=c_dtype))
    left = kappa[:, 0:1]
    right = torch.sum(kappa[:, 1:], dim=1, keepdim=True)
    return (torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)) ** 2


def weighted_normalized_mean(pointwise, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0):
    unweighted = torch.mean(pointwise)
    if adaptive_bar_weight:
        raw_w = (pointwise.detach() + bar_weight_eps) ** bar_weight_alpha
        weights = torch.clamp(raw_w / (torch.mean(raw_w) + bar_weight_eps), min=bar_weight_min, max=bar_weight_max).detach()
        weighted = torch.sum(weights * pointwise) / (torch.sum(weights) + bar_weight_eps)
    else:
        weights = torch.ones_like(pointwise).detach()
        weighted = unweighted
    return weighted, unweighted, weights


def compute_bar_loss(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    pointwise = compute_bar_pointwise(theta, f_tuple, params, eps_gamma=eps_gamma, eps_bar=eps_bar)
    weighted, unweighted, weights = weighted_normalized_mean(
        pointwise,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
    )
    if not return_diagnostics:
        return weighted
    return weighted, {
        "bar_unweighted": unweighted.detach(),
        "bar_weighted": weighted.detach(),
        "bar_weight_mean": torch.mean(weights).detach(),
        "bar_weight_min": torch.min(weights).detach(),
        "bar_weight_max": torch.max(weights).detach(),
    }


def boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma=1e-12, eps_bar=1e-8):
    gamma0, gamma_all = compute_gammas(theta_k, params)
    gammak = gamma_all[:, k:k+1]
    nu = torch.maximum(torch.log(torch.abs(gamma0) + eps_gamma) + f0.real, torch.log(torch.abs(gammak) + eps_gamma) + fk.real).to(dtype=c_dtype)
    left = gamma0 * torch.exp(f0 - nu)
    right = gammak * torch.exp(fk - nu)
    rel = torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)
    return rel ** 2


def compute_boundary_multiplicative_loss_generic(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f_mat = split_f_tuple(model(theta_k))
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f_mat[:, 0:1], f_mat[:, k+1:k+2], params, k, eps_gamma, eps_bar))
    return total / d


def compute_boundary_multiplicative_loss_structured(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f0, fk = model.forward_selected_boundary(theta_k, k)
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma, eps_bar))
    return total / d


def compute_zero_loss(model, params):
    theta0 = torch.zeros(1, params.R.shape[0], dtype=c_dtype, device=device)
    return torch.mean(torch.abs(torch.exp(model(theta0)[:, 0:1]) - 1.0) ** 2)


def compute_monotonicity_components_analytical_from_theta_real(model, theta_real):
    theta = torch.complex(theta_real.to(dtype=r_dtype, device=device), torch.zeros_like(theta_real.to(dtype=r_dtype, device=device)))
    f_mat, J = model.forward_with_input_jacobian(theta, need_jacobian=True)
    loss_mono = torch.mean(torch.relu(J.real) ** 2)
    loss_imag = torch.mean(f_mat.imag ** 2)
    return loss_mono, loss_imag


def compute_monotonicity_components_analytical(model, N_mono, d=20, re_min=-0.5, re_max=5.0):
    theta = sample_training_theta(N_mono, d=d, re_min=re_min, re_max=re_max, im_max=0.0, device=device)
    return compute_monotonicity_components_analytical_from_theta_real(model, theta.real.detach())


def compute_full_training_loss_structured(model, params, batch_bar=4000, batch_bdary=256, batch_mono=256, d=20, lambda_bar=1.0, lambda_bdary=1.0, lambda_zero=0.0, lambda_mono=1.0, lambda_imag=1.0, mono_re_min=-0.5, mono_re_max=5.0, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    theta_bar = sample_mixed_bar_theta(N=batch_bar, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device)
    loss_bar, bar_diag = compute_bar_loss(
        theta_bar,
        model(theta_bar),
        params,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
        return_diagnostics=True,
    )
    theta_bdary = sample_training_theta(batch_bdary, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device)
    loss_bdary = compute_boundary_multiplicative_loss_structured(model, theta_bdary, params)
    if lambda_zero == 0.0:
        loss_zero = torch.zeros((), dtype=r_dtype, device=device)
    else:
        loss_zero = compute_zero_loss(model, params)
    loss_mono, loss_imag = compute_monotonicity_components_analytical(model, batch_mono, d=d, re_min=mono_re_min, re_max=mono_re_max)
    total = lambda_bar*loss_bar + lambda_bdary*loss_bdary + lambda_zero*loss_zero + lambda_mono*loss_mono + lambda_imag*loss_imag
    if not return_diagnostics:
        return total
    diag = {"total_loss": total.detach(), "loss_bar": loss_bar.detach(), "loss_bar_unweighted": bar_diag["bar_unweighted"], "loss_bar_weighted": bar_diag["bar_weighted"], "loss_bdary": loss_bdary.detach(), "loss_zero": loss_zero.detach(), "loss_mono": loss_mono.detach(), "loss_imag": loss_imag.detach()}
    diag.update(bar_diag)
    return total, diag


## 7. Preflight, smoke test, and fixed-endpoint training

`train_one` is convenient for long laptop runs; `train_all` processes the three
seeds one after another, so peak accelerator memory is independent of the
number of seeds.  Only the step-25,000 endpoint is eligible for evaluation. The
learning-rate change preserves the same Adam state.


In [ ]:
LOG_COLUMNS = [
    "seed", "step", "learning_rate", "total_loss", "bar_loss",
    "bar_loss_unweighted", "boundary_loss", "monotonicity_loss",
    "imaginary_loss", "minimum_cauchy_denominator",
    "cumulative_sampled_constraints", "elapsed_seconds",
]


def gradients_are_finite(model):
    for parameter in model.parameters():
        if not parameter.requires_grad or parameter.grad is None:
            return False
        grad = parameter.grad
        if not torch.isfinite(grad.real if grad.is_complex() else grad).all():
            return False
        if grad.is_complex() and not torch.isfinite(grad.imag).all():
            return False
    return True


def loss_call(model, batch_bar, batch_boundary, batch_monotonicity):
    weight = TRAIN_CONFIG["adaptive_bar_weight"]
    return compute_full_training_loss_structured(
        model,
        PARAMS,
        batch_bar=batch_bar,
        batch_bdary=batch_boundary,
        batch_mono=batch_monotonicity,
        d=DIMENSION,
        lambda_bar=TRAIN_CONFIG["lambda_bar"],
        lambda_bdary=TRAIN_CONFIG["lambda_boundary"],
        lambda_zero=TRAIN_CONFIG["lambda_origin"],
        lambda_mono=TRAIN_CONFIG["lambda_monotonicity"],
        lambda_imag=TRAIN_CONFIG["lambda_imaginary"],
        mono_re_min=TRAIN_CONFIG["monotonicity_real_domain"][0],
        mono_re_max=TRAIN_CONFIG["monotonicity_real_domain"][1],
        adaptive_bar_weight=weight["enabled"],
        bar_weight_alpha=weight["alpha"],
        bar_weight_eps=weight["epsilon"],
        bar_weight_min=weight["minimum"],
        bar_weight_max=weight["maximum"],
        return_diagnostics=True,
    )


def run_preflight(seed=PAPER_SEEDS[0]):
    model = build_model(seed)
    model.train()
    theta = sample_training_theta(4, d=DIMENSION, device=device)
    f_values, jacobian = model.forward_with_input_jacobian(theta, need_jacobian=True)
    if f_values.shape != (4, DIMENSION + 1):
        raise AssertionError(f"Unexpected model shape {tuple(f_values.shape)}")
    if jacobian.shape != (4, DIMENSION + 1, DIMENSION):
        raise AssertionError(f"Unexpected Jacobian shape {tuple(jacobian.shape)}")

    zero = torch.zeros(2, DIMENSION, dtype=c_dtype, device=device)
    with torch.no_grad():
        f_zero = model(zero)
    interior_origin_error = float(torch.max(torch.abs(f_zero[:, 0])).detach().cpu())
    boundary_anchor_error = float(torch.max(
        torch.abs(f_zero[:, 1:] - torch.log(BOUNDARY_MASS).to(c_dtype))
    ).detach().cpu())

    reset_all_cauchy_denominator_tracking(model)
    total, diagnostics = loss_call(model, 16, 4, 4)
    if not torch.isfinite(total):
        raise FloatingPointError("Preflight loss is non-finite.")
    total.backward()
    finite_gradients = gradients_are_finite(model)
    minimum_denominator = min_cauchy_denominator(model)

    report = {
        "seed": int(seed),
        "output_shape": list(f_values.shape),
        "jacobian_shape": list(jacobian.shape),
        "jacobian_finite": bool(torch.isfinite(jacobian.real).all() and torch.isfinite(jacobian.imag).all()),
        "loss_terms_finite": bool(all(
            torch.isfinite(value).all().item()
            for value in diagnostics.values() if torch.is_tensor(value)
        )),
        "gradients_present_and_finite": bool(finite_gradients),
        "interior_origin_error": interior_origin_error,
        "boundary_anchor_error": boundary_anchor_error,
        "minimum_cauchy_denominator": minimum_denominator,
        "parameter_real_scalar_dof": PARAMETER_AUDIT["trainable_real_scalar_dof"],
    }
    if (
        not report["jacobian_finite"]
        or not report["loss_terms_finite"]
        or not finite_gradients
        or interior_origin_error > 1.0e-11
        or boundary_anchor_error > 1.0e-11
        or not math.isfinite(minimum_denominator)
        or minimum_denominator <= 0.0
    ):
        raise AssertionError(report)
    return report


def optimizer_step(model, optimizer, batch_bar, batch_boundary, batch_monotonicity):
    reset_all_cauchy_denominator_tracking(model)
    optimizer.zero_grad(set_to_none=True)
    total, diagnostics = loss_call(model, batch_bar, batch_boundary, batch_monotonicity)
    if not torch.isfinite(total):
        raise FloatingPointError("Training loss is non-finite.")
    total.backward()
    if not gradients_are_finite(model):
        raise FloatingPointError("A trainable parameter has a missing or non-finite gradient.")
    torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG["gradient_clip"])
    optimizer.step()
    return diagnostics, min_cauchy_denominator(model)


def run_smoke_test():
    preflight = run_preflight(PAPER_SEEDS[0])
    model = build_model(PAPER_SEEDS[0])
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_phase1"])
    losses = []
    for _ in range(2):
        diagnostics, _ = optimizer_step(model, optimizer, 16, 4, 4)
        losses.append(float(diagnostics["total_loss"].detach().cpu()))
    result = {"preflight": preflight, "two_step_losses": losses, "passed": True}
    print(json.dumps(result, indent=2))
    return result


def save_training_plot(rows, path):
    steps = [row["step"] for row in rows]
    plt.figure(figsize=(9.0, 5.5))
    plt.semilogy(steps, [row["total_loss"] for row in rows], label="total")
    plt.semilogy(steps, [row["bar_loss"] for row in rows], label="BAR")
    plt.semilogy(steps, [row["boundary_loss"] for row in rows], label="boundary")
    plt.axvline(LR_SWITCH_AFTER_STEP, color="black", linestyle="--", label="learning-rate switch")
    plt.xlabel("optimizer step")
    plt.ylabel("loss")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def train_one_seed(seed, run_dir):
    seed = int(seed)
    seed_dir = Path(run_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=False)
    set_all_seeds(seed)
    model = build_model(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_phase1"])
    rows = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()

    for step in range(1, TRAIN_STEPS + 1):
        if step == LR_SWITCH_AFTER_STEP + 1:
            for group in optimizer.param_groups:
                group["lr"] = TRAIN_CONFIG["lr_phase2"]

        diagnostics, minimum_denominator = optimizer_step(
            model,
            optimizer,
            TRAIN_CONFIG["batch_bar"],
            TRAIN_CONFIG["batch_boundary"],
            TRAIN_CONFIG["batch_monotonicity"],
        )

        if step % LOG_EVERY == 0 or step == TRAIN_STEPS:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            row = {
                "seed": seed,
                "step": step,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "total_loss": float(diagnostics["total_loss"].detach().cpu()),
                "bar_loss": float(diagnostics["loss_bar"].detach().cpu()),
                "bar_loss_unweighted": float(diagnostics["loss_bar_unweighted"].detach().cpu()),
                "boundary_loss": float(diagnostics["loss_bdary"].detach().cpu()),
                "monotonicity_loss": float(diagnostics["loss_mono"].detach().cpu()),
                "imaginary_loss": float(diagnostics["loss_imag"].detach().cpu()),
                "minimum_cauchy_denominator": minimum_denominator,
                "cumulative_sampled_constraints": step * (
                    TRAIN_CONFIG["batch_bar"]
                    + TRAIN_CONFIG["batch_boundary"]
                    + TRAIN_CONFIG["batch_monotonicity"]
                ),
                "elapsed_seconds": time.perf_counter() - start,
            }
            rows.append(row)
            print(row)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak_allocated = int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0
    peak_reserved = int(torch.cuda.max_memory_reserved()) if torch.cuda.is_available() else 0

    checkpoint = seed_dir / f"checkpoint_step{CHECKPOINT_STEP}.pt"
    payload = {
        "experiment_slug": EXPERIMENT_SLUG,
        "seed": seed,
        "last_completed_step": TRAIN_STEPS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": MODEL_CONFIG,
        "training_config": TRAIN_CONFIG,
        "sampler_config": SAMPLER_CONFIG,
        "parameter_audit": PARAMETER_AUDIT,
        "boundary_mass": BOUNDARY_MASS.detach().cpu(),
    }
    torch.save(payload, checkpoint)
    checkpoint_hash = sha256_file(checkpoint)

    write_csv(seed_dir / "training_log.csv", rows, LOG_COLUMNS)
    save_training_plot(rows, seed_dir / "training_loss.png")
    write_json(seed_dir / "training_runtime.json", {
        "seed": seed,
        "training_wall_clock_seconds": elapsed,
        "peak_gpu_allocated_bytes": peak_allocated,
        "peak_gpu_reserved_bytes": peak_reserved,
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": checkpoint_hash,
    })
    return checkpoint


def new_run_directory():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    run_dir = OUTPUT_ROOT / f"{EXPERIMENT_SLUG}_{utc_stamp()}"
    run_dir.mkdir(parents=False, exist_ok=False)
    return run_dir


SMOKE_REPORT = None
FORMAL_RUN_DIR = None
if RUN_MODE == "smoke":
    SMOKE_REPORT = run_smoke_test()
elif RUN_MODE in {"train_one", "train_all"}:
    PREFLIGHT_AUDIT = run_preflight(PAPER_SEEDS[0])
    FORMAL_RUN_DIR = new_run_directory()
    ACTIVE_TRAIN_SEEDS = (TRAIN_SEED,) if RUN_MODE == "train_one" else PAPER_SEEDS
    ACTIVE_RUN_CONFIG = dict(RUN_CONFIG)
    ACTIVE_RUN_CONFIG["active_training_seeds"] = list(ACTIVE_TRAIN_SEEDS)
    ACTIVE_SAMPLE_BUDGET = dict(SAMPLE_BUDGET)
    ACTIVE_SAMPLE_BUDGET["active_training_seeds"] = list(ACTIVE_TRAIN_SEEDS)
    ACTIVE_SAMPLE_BUDGET["active_run_total_sampled_constraints"] = (
        len(ACTIVE_TRAIN_SEEDS) * PER_SEED_SAMPLE_BUDGET
    )
    write_json(FORMAL_RUN_DIR / "run_config.json", ACTIVE_RUN_CONFIG)
    write_json(FORMAL_RUN_DIR / "rbm_audit.json", RBM_AUDIT)
    write_json(FORMAL_RUN_DIR / "parameter_budget_audit.json", PARAMETER_AUDIT)
    write_json(FORMAL_RUN_DIR / "sampler_audit.json", {
        "configuration": SAMPLER_CONFIG,
        "actual_counts_per_bar_update": ACTUAL_BAR_MIXTURE_COUNTS,
    })
    write_json(FORMAL_RUN_DIR / "sample_budget.json", ACTIVE_SAMPLE_BUDGET)
    write_json(FORMAL_RUN_DIR / "preflight_audit.json", PREFLIGHT_AUDIT)
    write_json(FORMAL_RUN_DIR / "environment.json", {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    })
    for paper_seed in ACTIVE_TRAIN_SEEDS:
        train_one_seed(paper_seed, FORMAL_RUN_DIR)
    print("Fixed endpoints written to", FORMAL_RUN_DIR)


## 8. Post-training Talbot evaluation

Ground truth first appears in this cell.  Degree 5 is retained to connect to
the published low-degree convention and to quantify its inversion floor.
Degree 8 is the primary 20D metric because applying Degree 5 even to the exact
transform produces a material tail-inversion error.  Both degrees evaluate the
same frozen step-25,000 checkpoints; neither degree affects training or model
selection.


In [ ]:
# Ground truth and published comparison data are deliberately defined only now.
mp.mp.dps = EVALUATION_CONFIG["mpmath_decimal_digits"]
T_VALUES = EVALUATION_CONFIG["t_values"]
TALBOT_DEGREES = EVALUATION_CONFIG["talbot_degrees"]
RATES = [mp.mpf(index) for index in range(1, DIMENSION + 1)]


def exact_product_form_transform(s):
    s = mp.mpc(s)
    value = mp.mpf("1")
    for rate in RATES:
        value *= rate / (rate + s)
    return value


def exact_tail_probability(t):
    t = mp.mpf(t)
    total = mp.mpf("0")
    for i, rate_i in enumerate(RATES):
        coefficient = mp.mpf("1")
        for j, rate_j in enumerate(RATES):
            if i != j:
                coefficient *= rate_j / (rate_j - rate_i)
        total += coefficient * mp.exp(-rate_i * t)
    return total


TRUE_TAIL = {int(t): float(exact_tail_probability(t)) for t in T_VALUES}

# Published NN4MGF Degree-5 table for the same 20D benchmark.
PUBLISHED_NN4MGF_DEGREE5 = {
    1: 1.000628,
    2: 0.9646730,
    3: 0.6027136,
    4: 0.2814302,
    5: 0.1223421,
    6: 0.05247283,
    7: 0.02165455,
    8: 0.007748090,
    9: 0.001973230,
    10: 0.0001452148,
}
NN4MGF_PROVENANCE = {
    "repository": "https://github.com/zhangz73/NN4MGF",
    "observed_commit": "1956666902281f6ce4558ca82d8d15fa038f06fb",
    "published_table_degree": 5,
    "released_20d_checkpoint_sha256": "5dfa26e2fe9e44c7726c2a56545be52e1bdac7a6ae14ab057f359f008fede67e",
    "note": "The embedded values are the published Degree-5 table; the released-checkpoint Degree-8 comparison belongs to the separate controlled-comparison artifact.",
}


def learned_diagonal_zero_limit(model):
    zero = torch.zeros(1, DIMENSION, dtype=c_dtype, device=device)
    with torch.no_grad():
        _, jacobian = model.forward_with_input_jacobian(zero, need_jacobian=True)
    value = -torch.sum(jacobian[0, 0, :])
    return mp.mpc(float(value.real.detach().cpu()), float(value.imag.detach().cpu()))


def learned_transform_adapter(model):
    zero_limit = learned_diagonal_zero_limit(model)

    @torch.no_grad()
    def phi(s):
        z = mp.mpc(s)
        theta = torch.full(
            (1, DIMENSION),
            complex(float(mp.re(z)), float(mp.im(z))),
            dtype=c_dtype,
            device=device,
        )
        value = torch.exp(model(theta)[0, 0])
        return mp.mpc(float(value.real.detach().cpu()), float(value.imag.detach().cpu()))

    def tail_transform(s):
        s = mp.mpc(s)
        if s == 0:
            return zero_limit
        return (1 - phi(s)) / s

    return tail_transform, zero_limit


def exact_tail_transform(s):
    s = mp.mpc(s)
    if s == 0:
        return mp.fsum(1 / rate for rate in RATES)
    return (1 - exact_product_form_transform(s)) / s


def parse_prediction(value):
    value = mp.mpc(value)
    real = float(mp.re(value))
    imaginary = float(mp.im(value))
    invalid = (
        not math.isfinite(real)
        or not math.isfinite(imaginary)
        or abs(imaginary) > 1.0e-8
        or real < 0.0
        or real > 1.0
    )
    return real, imaginary, bool(invalid)


def relative_error(prediction, truth):
    return abs(prediction - truth) / abs(truth)


def interval_mre(rows, end_t):
    selected = [row for row in rows if int(row["t"]) <= int(end_t)]
    # MRE is computed from raw finite values even when a value lies outside
    # [0,1]; the separate invalid count makes that event visible without
    # silently clipping or discarding the prediction.
    return float(np.mean([row["relative_error"] for row in selected]))


def evaluate_exact_floor():
    rows = []
    for degree in TALBOT_DEGREES:
        for t in T_VALUES:
            raw = mp.invertlaplace(exact_tail_transform, mp.mpf(t), method="talbot", degree=degree)
            prediction, imaginary, invalid = parse_prediction(raw)
            truth = TRUE_TAIL[t]
            rows.append({
                "method": "exact_transform_floor",
                "degree": degree,
                "t": t,
                "true_value": truth,
                "raw_prediction": prediction,
                "raw_prediction_imaginary": imaginary,
                "relative_error": relative_error(prediction, truth),
                "invalid_flag": invalid,
            })
    return rows


def load_fixed_model(checkpoint_path, expected_seed):
    payload = torch.load(checkpoint_path, map_location=device, weights_only=True)
    if int(payload.get("last_completed_step", -1)) != CHECKPOINT_STEP:
        raise RuntimeError(f"Expected fixed step {CHECKPOINT_STEP}: {checkpoint_path}")
    if int(payload.get("seed", -1)) != int(expected_seed):
        raise RuntimeError(f"Checkpoint seed mismatch at {checkpoint_path}")
    if payload.get("model_config") != MODEL_CONFIG:
        raise RuntimeError(f"Model configuration mismatch at {checkpoint_path}")
    model = build_model(expected_seed)
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.eval()
    return model


def evaluate_seed(checkpoint_path, seed):
    checkpoint_path = Path(checkpoint_path)
    hash_before = sha256_file(checkpoint_path)
    model = load_fixed_model(checkpoint_path, seed)
    transform, zero_limit = learned_transform_adapter(model)
    rows = []
    started = time.perf_counter()
    for degree in TALBOT_DEGREES:
        for t in T_VALUES:
            raw = mp.invertlaplace(transform, mp.mpf(t), method="talbot", degree=degree)
            prediction, imaginary, invalid = parse_prediction(raw)
            truth = TRUE_TAIL[t]
            rows.append({
                "benchmark": EXPERIMENT_SLUG,
                "seed": int(seed),
                "step": CHECKPOINT_STEP,
                "degree": degree,
                "t": t,
                "true_value": truth,
                "raw_prediction": prediction,
                "raw_prediction_imaginary": imaginary,
                "relative_error": relative_error(prediction, truth),
                "invalid_flag": invalid,
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_sha256": hash_before,
            })
    elapsed = time.perf_counter() - started
    hash_after = sha256_file(checkpoint_path)
    if hash_before != hash_after:
        raise AssertionError(f"Evaluation modified checkpoint {checkpoint_path}")
    summary = []
    for degree in TALBOT_DEGREES:
        selected = [row for row in rows if row["degree"] == degree]
        summary.append({
            "benchmark": EXPERIMENT_SLUG,
            "seed": int(seed),
            "step": CHECKPOINT_STEP,
            "degree": degree,
            "mre_1_5": interval_mre(selected, 5),
            "mre_1_8": interval_mre(selected, 8),
            "mre_1_10": interval_mre(selected, 10),
            "invalid_count_1_10": int(sum(row["invalid_flag"] for row in selected)),
            "evaluation_seconds": elapsed,
            "learned_zero_limit_real": float(mp.re(zero_limit)),
            "learned_zero_limit_imaginary": float(mp.im(zero_limit)),
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_sha256": hash_before,
        })
    return rows, summary


def save_evaluation_plots(run_dir, pointwise_rows, evaluated_seeds):
    for degree in TALBOT_DEGREES:
        degree_rows = [row for row in pointwise_rows if row["degree"] == degree]
        plt.figure(figsize=(9.0, 5.5))
        plt.semilogy(T_VALUES, [TRUE_TAIL[t] for t in T_VALUES], "k-o", label="exact tail")
        for seed in evaluated_seeds:
            rows = [row for row in degree_rows if row["seed"] == seed]
            plotted = [
                row["raw_prediction"] if not row["invalid_flag"] and row["raw_prediction"] > 0 else np.nan
                for row in rows
            ]
            plt.semilogy(T_VALUES, plotted, marker="x", label=f"HCN seed {seed}")
        if degree == 5:
            plt.semilogy(
                T_VALUES,
                [PUBLISHED_NN4MGF_DEGREE5[t] for t in T_VALUES],
                marker="^",
                label="published NN4MGF (Degree 5)",
            )
        plt.xlabel("t")
        plt.ylabel("tail probability (raw)")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(Path(run_dir) / f"tail_probability_degree{degree}.png", dpi=180)
        plt.close()

        plt.figure(figsize=(9.0, 5.5))
        for seed in evaluated_seeds:
            rows = [row for row in degree_rows if row["seed"] == seed]
            plt.semilogy(T_VALUES, [row["relative_error"] for row in rows], marker="o", label=f"HCN seed {seed}")
        plt.xlabel("t")
        plt.ylabel("raw relative error")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(Path(run_dir) / f"relative_error_degree{degree}.png", dpi=180)
        plt.close()


def evaluate_run(run_dir, evaluated_seeds=PAPER_SEEDS):
    run_dir = Path(run_dir).expanduser().resolve()
    evaluated_seeds = tuple(int(seed) for seed in evaluated_seeds)
    pointwise_rows = []
    summary_rows = []
    for seed in evaluated_seeds:
        checkpoint = run_dir / f"seed_{seed}" / f"checkpoint_step{CHECKPOINT_STEP}.pt"
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Missing fixed endpoint: {checkpoint}")
        rows, summaries = evaluate_seed(checkpoint, seed)
        pointwise_rows.extend(rows)
        summary_rows.extend(summaries)

    aggregate_rows = []
    for degree in TALBOT_DEGREES:
        selected = [row for row in summary_rows if row["degree"] == degree]
        for metric in ("mre_1_5", "mre_1_8", "mre_1_10"):
            values = np.asarray([row[metric] for row in selected], dtype=float)
            aggregate_rows.append({
                "degree": degree,
                "metric": metric,
                "number_of_seeds": len(values),
                "seeds": ";".join(map(str, evaluated_seeds)),
                "mean": float(np.mean(values)),
                "population_standard_deviation": float(np.std(values, ddof=0)),
                "minimum": float(np.min(values)),
                "maximum": float(np.max(values)),
                "invalid_count_total_1_10": int(sum(row["invalid_count_1_10"] for row in selected)),
            })

    exact_floor_rows = evaluate_exact_floor()
    exact_floor_summary = []
    for degree in TALBOT_DEGREES:
        rows = [row for row in exact_floor_rows if row["degree"] == degree]
        exact_floor_summary.append({
            "degree": degree,
            "mre_1_5": interval_mre(rows, 5),
            "mre_1_8": interval_mre(rows, 8),
            "mre_1_10": interval_mre(rows, 10),
            "invalid_count_1_10": int(sum(row["invalid_flag"] for row in rows)),
        })

    official_rows = []
    for t in T_VALUES:
        prediction = PUBLISHED_NN4MGF_DEGREE5[t]
        truth = TRUE_TAIL[t]
        official_rows.append({
            "degree": 5,
            "t": t,
            "true_value": truth,
            "raw_prediction": prediction,
            "relative_error": relative_error(prediction, truth),
            "invalid_flag": bool(not math.isfinite(prediction) or prediction < 0.0 or prediction > 1.0),
        })

    write_csv(run_dir / "checkpoint_pointwise.csv", pointwise_rows)
    write_csv(run_dir / "checkpoint_summary.csv", summary_rows)
    write_csv(run_dir / "multiseed_aggregate.csv", aggregate_rows)
    write_csv(run_dir / "exact_transform_talbot_floor.csv", exact_floor_rows)
    write_csv(run_dir / "exact_transform_talbot_floor_summary.csv", exact_floor_summary)
    write_csv(run_dir / "published_nn4mgf_degree5.csv", official_rows)
    write_json(run_dir / "official_baseline_provenance.json", NN4MGF_PROVENANCE)
    save_evaluation_plots(run_dir, pointwise_rows, evaluated_seeds)

    final_summary = {
        "primary_endpoint": f"fixed step {CHECKPOINT_STEP}",
        "paper_seeds": list(PAPER_SEEDS),
        "evaluated_seeds": list(evaluated_seeds),
        "primary_talbot_degree": EVALUATION_CONFIG["primary_degree"],
        "ground_truth_used_in_training": False,
        "ground_truth_used_for_checkpoint_selection": False,
        "raw_predictions": True,
        "prediction_clipping": False,
        "parameter_real_scalar_dof": PARAMETER_AUDIT["trainable_real_scalar_dof"],
        "per_seed_total_sampled_constraints": PER_SEED_SAMPLE_BUDGET,
        "multiseed_aggregate": aggregate_rows,
        "exact_transform_floor": exact_floor_summary,
        "published_nn4mgf_degree5_mre_1_10": interval_mre(official_rows, 10),
    }
    write_json(run_dir / "evaluation_summary.json", final_summary)
    print(json.dumps(final_summary, indent=2))
    return final_summary


EVALUATION_SUMMARY = None
if RUN_MODE in {"train_one", "train_all"}:
    EVALUATION_SUMMARY = evaluate_run(FORMAL_RUN_DIR, ACTIVE_TRAIN_SEEDS)
elif RUN_MODE == "evaluate":
    EVALUATION_SUMMARY = evaluate_run(EVALUATE_RUN_DIR)
else:
    print("Smoke test complete. No ground truth was used and no formal results were written.")
